In [2]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime

# ==============================
# 1. Cấu hình chung
# ==============================
BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
START_DATE = "20190101"
END_DATE = datetime.today().strftime("%Y%m%d")

COMMON_PARAMS = {
    "community": "AG",
    "format": "JSON",
    "start": START_DATE,
    "end": END_DATE
}

# ==============================
# 2. Bounding box quốc gia
# ==============================
# COUNTRIES = {
#     "mexico": {"lat": (14, 33), "lon": (-118, -86)},
#     "bangladesh": {"lat": (20, 27), "lon": (88, 93)},
#     "malaysia": {"lat": (0.8, 7.5), "lon": (99, 120)},
#     "philippines": {"lat": (5, 19), "lon": (116, 127)},
#     "pakistan": {"lat": (23, 37), "lon": (60, 77)},
# }
COUNTRIES = {
    "egypt": {"lat": (22, 32), "lon": (25, 36)},            # Ai Cập
    "south_africa": {"lat": (-35, -22), "lon": (16, 33)},  # Nam Phi
    "colombia": {"lat": (-4, 13), "lon": (-79, -66)},      # Colombia
    "peru": {"lat": (-18, 0), "lon": (-82, -68)},          # Peru
    "argentina": {"lat": (-55, -21), "lon": (-73, -53)},   # Argentina
}

# ==============================
# 3. Đặc trưng & aggregation rule
# ==============================
FEATURES = {
    "temperature": {"param": "T2M", "agg": "mean"},
    "humidity": {"param": "QV2M", "agg": "mean"},
    "solar_radiation": {"param": "ALLSKY_SFC_SW_DWN", "agg": "mean"},
    "precipitation": {"param": "PRECTOTCORR", "agg": "sum"},
}

# ==============================
# 4. Tạo grid điểm
# ==============================
def create_grid(lat_range, lon_range, step=2.0):
    lats = np.arange(lat_range[0], lat_range[1] + step, step)
    lons = np.arange(lon_range[0], lon_range[1] + step, step)
    return [(lat, lon) for lat in lats for lon in lons]

# ==============================
# 5. Fetch daily data 1 điểm
# ==============================
def fetch_daily(lat, lon, parameter):
    params = COMMON_PARAMS.copy()
    params.update({
        "latitude": lat,
        "longitude": lon,
        "parameters": parameter
    })

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()

    data = r.json()["properties"]["parameter"][parameter]
    df = pd.DataFrame.from_dict(data, orient="index", columns=[parameter])
    df.index = pd.to_datetime(df.index)
    return df

# ==============================
# 6. Spatial average → Monthly EOM
# ==============================
def spatial_monthly_average(dfs, agg):
    combined = pd.concat(dfs, axis=1)

    if agg == "mean":
        daily_avg = combined.mean(axis=1)
    elif agg == "sum":
        daily_avg = combined.sum(axis=1)
    else:
        raise ValueError("Invalid aggregation")

    monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")
    monthly = monthly.reset_index()
    monthly.columns = ["date", "value"]
    return monthly

# ==============================
# 7. Pipeline chính
# ==============================
for country, bbox in COUNTRIES.items():
    print(f"\n🌍 Processing {country.upper()}")

    grid_points = create_grid(bbox["lat"], bbox["lon"], step=2.5)
    print(f"  Grid points: {len(grid_points)}")

    for feature, meta in FEATURES.items():
        daily_dfs = []

        for lat, lon in grid_points:
            try:
                df = fetch_daily(lat, lon, meta["param"])
                daily_dfs.append(df)
            except:
                continue

        monthly_df = spatial_monthly_average(daily_dfs, meta["agg"])
        filename = f"{country}_{feature}_monthly_country.csv"
        monthly_df.to_csv(filename, index=False)

        print(f"  ✔ Saved: {filename}")


🌍 Processing EGYPT
  Grid points: 30


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\414997120.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")


  ✔ Saved: egypt_temperature_monthly_country.csv


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\414997120.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")


  ✔ Saved: egypt_humidity_monthly_country.csv


...

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\414997120.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")


  ✔ Saved: argentina_humidity_monthly_country.csv


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\414997120.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")


  ✔ Saved: argentina_solar_radiation_monthly_country.csv
  ✔ Saved: argentina_precipitation_monthly_country.csv


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\414997120.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = daily_avg.resample("M").agg("mean" if agg == "mean" else "sum")
